In [2]:
#Loading the Excel File and libraries
import pandas as pd
import numpy as np

file_path = '/content/Global Oil and Gas Extraction Tracker - 2024-02-22_1039.xlsx'
xl = pd.ExcelFile(file_path)

In [3]:
#setting dataframes with main, P&R
main_data_df = xl.parse('Main data')
production_reserves_df = xl.parse('Production & reserves')


In [4]:
# Ensure "FID Year" and "Production start year" are in numeric format
main_data_df["FID Year"] = main_data_df["FID Year"].astype(str).str.replace(" \(expected\)", "", regex=True)
main_data_df["FID Year"] = pd.to_numeric(main_data_df["FID Year"], errors='coerce')
main_data_df["Production start year"] = pd.to_numeric(main_data_df["Production start year"], errors='coerce')

# Initialize a new column for marking the rows
main_data_df["Marking"] = ""

# Mark rows based on the specific condition
condition = main_data_df["FID Year"].isna() & (main_data_df["Production start year"] >= 2024)
main_data_df.loc[condition, "Marking"] = "Included based on production year"

# Filtering and naming the DataFrame as filtered_main_data_df
filtered_main_data_df = main_data_df[
    (main_data_df["FID Year"] >= 2024) |
    condition
]


In [5]:
# Convert 'Data year' to integers in the 'production_reserves_df'
production_reserves_df['Data year'] = pd.to_numeric(production_reserves_df['Data year'], errors='coerce').fillna(-1).astype(int)


In [6]:
#Define functions for getting the most recent value, calculate the total reserves, calculate the total production and define the conversion factor
# Function to get the most recent value based on the criteria
def get_most_recent_value(unit_id, prod_res, units, df):
    # Filter based on Unit ID, Production/reserves, and Units (converted)
    filtered = df[
        (df["Unit ID"] == unit_id) &
        (df["Production/reserves"] == prod_res) &
        (df["Units (converted)"] == units)
    ]

    # Sort by Data year and get the most recent entry
    filtered = filtered.sort_values(by="Data year", ascending=False)
    return filtered.iloc[0]["Quantity (converted)"] if not filtered.empty else np.nan

# Function to calculate the total reserves
def calculate_total_reserves(row):
    if pd.notna(row['Reserves- Hydrocarbons (unspecified)']):
        return row['Reserves- Hydrocarbons (unspecified)']
    else:
        # Convert gas reserves to boe
        gas_in_boe = row['Reserves- Gas'] * conversion_factor if pd.notna(row['Reserves- Gas']) else 0
        oil_reserves = row['Reserves - Oil'] if pd.notna(row['Reserves - Oil']) else 0
        return gas_in_boe + oil_reserves

# Function to calculate the total production
def calculate_total_production(row):
    if pd.notna(row['Production - Hydrocarbons (unspecified)']):
        return row['Production - Hydrocarbons (unspecified)']
    else:
        # Convert gas production to boe
        gas_in_boe = row['Production - Gas'] * conversion_factor if pd.notna(row['Production - Gas']) else 0
        oil_production = row['Production - Oil'] if pd.notna(row['Production - Oil']) else 0
        return gas_in_boe + oil_production

# Conversion factor from million m³ to million boe for gas
conversion_factor = 5.883 / 1000


In [7]:
# Populate reserves and production columns with the most recent data
filtered_main_data_df["Reserves - Oil"] = filtered_main_data_df["Unit ID"].apply(lambda x: get_most_recent_value(x, "reserves", "million bbl", production_reserves_df))
filtered_main_data_df["Reserves- Gas"] = filtered_main_data_df["Unit ID"].apply(lambda x: get_most_recent_value(x, "reserves", "million m³", production_reserves_df))
filtered_main_data_df["Reserves- Hydrocarbons (unspecified)"] = filtered_main_data_df["Unit ID"].apply(lambda x: get_most_recent_value(x, "reserves", "million boe", production_reserves_df))

filtered_main_data_df["Production - Oil"] = filtered_main_data_df["Unit ID"].apply(lambda x: get_most_recent_value(x, "production", "million bbl/y", production_reserves_df))
filtered_main_data_df["Production - Gas"] = filtered_main_data_df["Unit ID"].apply(lambda x: get_most_recent_value(x, "production", "million m³/y", production_reserves_df))
filtered_main_data_df["Production - Hydrocarbons (unspecified)"] = filtered_main_data_df["Unit ID"].apply(lambda x: get_most_recent_value(x, "production", "million boe/y", production_reserves_df))

# Calculate total reserves and production
filtered_main_data_df['Reserves- Total (Oil, Gas and Hydrocarbons)'] = filtered_main_data_df.apply(calculate_total_reserves, axis=1)
filtered_main_data_df['Production - Total (Oil, Gas and Hydrocarbons)'] = filtered_main_data_df.apply(calculate_total_production, axis=1)


<ipython-input-7-e0a58fe69be6>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_main_data_df["Reserves - Oil"] = filtered_main_data_df["Unit ID"].apply(lambda x: get_most_recent_value(x, "reserves", "million bbl", production_reserves_df))
<ipython-input-7-e0a58fe69be6>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_main_data_df["Reserves- Gas"] = filtered_main_data_df["Unit ID"].apply(lambda x: get_most_recent_value(x, "reserves", "million m³", production_reserves_df))
<ipython-input-7-e

In [8]:
filtered_main_data_df

,Unit ID,Unit Name,Unit name local script,Fuel type,Unit type,Country,"Subnational unit (province, state)",Latitude,Longitude,Location accuracy,...,Wiki URL,Marking,Reserves - Oil,Reserves- Gas,Reserves- Hydrocarbons (unspecified),Production - Oil,Production - Gas,Production - Hydrocarbons (unspecified),"Reserves- Total (Oil, Gas and Hydrocarbons)","Production - Total (Oil, Gas and Hydrocarbons)"
83,OG0000471,Aphrodite,NaN,oil and gas,field,Cyprus,NaN,33.184677,33.341657,exact,...,https://www.gem.wiki/Aphrodite_Oil_and_Gas_Fie...,,7.00,102506.017300,NaN,NaN,NaN,NaN,610.042900,0.0
1010,OG0002673,Zama,Zama,oil and gas,field,Mexico,NaN,18.893111,-93.608627,approximate,...,https://www.gem.wiki/Zama_Oil_and_Gas_Field_%2...,,486.44,5364.859125,NaN,NaN,NaN,NaN,518.001466,0.0
1477,OG0004886,Adda,NaN,oil and gas,field,Denmark,NaN,55.806300,4.863300,exact,...,https://www.gem.wiki/Adda_Oil_and_Gas_Field_%2...,,16.90,5711.371749,NaN,NaN,NaN,NaN,50.500000,0.0
2234,OG0010211,Dorado,NaN,oil and gas,field,Australia,Western Australia,-19.027222,118.743539,exact,...,https://www.gem.wiki/Dorado_Oil_and_Gas_Field_...,,162.00,21180.801360,NaN,NaN,NaN,NaN,286.606654,0.0
2363,OG0010818,Equus Gas,NaN,oil and gas,project,Australia,Western Australia,-20.110000,113.790000,approximate,...,https://www.gem.wiki/Equus_Oil_and_Gas_Project...,,42.00,56633.158710,NaN,NaN,NaN,NaN,375.172873,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8244,OG0020348,Sagitario,NaN,oil and gas,field,Brazil,NaN,-23.629739,-43.997817,approximate,...,https://www.gem.wiki/Sagitario_Oil_and_Gas_Fie...,,423.63,NaN,NaN,NaN,NaN,NaN,423.630000,0.0
8245,OG0020350,Browse to North West Shelf,NaN,oil and gas,project,Australia,Western Australia,-13.990000,121.980000,approximate,...,https://www.gem.wiki/Browse_to_North_West_Shel...,,390.00,393600.453100,NaN,NaN,NaN,NaN,2705.551466,0.0
8305,OG0020620,Elk-Antelope,NaN,gas,field,Papua New Guinea,NaN,-7.432792,144.911287,approximate,...,https://www.gem.wiki/Elk-Antelope_Gas_Field_%2...,,57.00,137335.409900,NaN,NaN,NaN,NaN,864.944216,0.0
8320,OG0020654,Kudu,NaN,gas,field,Namibia,NaN,-28.535155,14.662271,exact,...,https://www.gem.wiki/Kudu_Gas_Field_%28Namibia%29,,NaN,36811.553160,NaN,NaN,NaN,NaN,216.562367,0.0


In [9]:
output_excel_path = '/content/FIDafter2024 20240222.xlsx'
filtered_main_data_df.to_excel(output_excel_path, index=False)